## Python For Machine Learning Fall 2025
---
# Sequence-to-Sequence Models

# 1. Overview

We will use the terms `Seq2Seq` and `encoder-decoder` interchangeably, though they are different. The core concepts are as follows:
1.  **The Encoder:** An RNN or LSTM that reads an input sequence (e.g., an English sentence) token by token. Its job is to compress the entire meaning of the input into a single vector.
2.  **The Context Vector:** This is the final hidden state of the Encoder. It represents the "bottleneck" of the architecture—the entire input sequence must be summarized into this one numerical representation.
3.  **The Decoder:** Another RNN/LSTM that takes the context vector and generates an output sequence (e.g., a French sentence), one token at a time.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np

SEED = 73
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


# 2. Implementation

We implement a sequence-to-sequence model here to demonstrate the concept more clearly.

## 2.1 Data Preparation

We will use a small, synthetic dataset. In a real-world scenario, you would use large datasets and training will takes hours.


In [ ]:
raw_data = [
    ("i am cold", "j'ai froid"),
    ("you are kind", "tu es gentil"),
    ("he is here", "il est ici"),
    ("she is happy", "elle est heureuse"),
    ("we are friends", "nous sommes amis"),
    ("they are tired", "ils sont fatigués"),
    ("i am hungry", "j'ai faim"),
    ("you are tall", "tu es grand"),
    ("he is short", "il est petit"),
    ("she is smart", "elle est intelligente")
]


We also need to handle **Tokenization** (splitting strings into words) and **Vocabulary Building** (converting words to numerical IDs).

We will define special tokens:
* `<SOS>`: Start Of Sentence (tells the decoder to start generating).
* `<EOS>`: End Of Sentence (tells the model the sentence is finished).

In [ ]:
SOS_token = 0
EOS_token = 1

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

input_lang = Lang('eng')
output_lang = Lang('fra')

pairs = []
for eng, fra in raw_data:
    input_lang.addSentence(eng)
    output_lang.addSentence(fra)
    pairs.append((eng, fra))

print(f"English words: {input_lang.n_words}")
print(f"French words: {output_lang.n_words}")
print(f"Sample pair: {random.choice(pairs)}")

English words: 21
French words: 22
Sample pair: ('we are friends', 'nous sommes amis')


Helper function to convert sentences to tensors.

In [ ]:
def tensorFromSentence(lang, sentence):
    indexes = [lang.word2index[word] for word in sentence.split(' ')]
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

## 2.2 The Encoder

The Encoder is an LSTM (Long Short-Term Memory) network.

1.  **Input:** It takes one word index at a time.
2.  **Embedding:** Converts the index into a dense vector (Embedding Layer).
3.  **LSTM Layer:** Updates its hidden state based on the current word and the previous state.
4.  **Output:** We mostly care about the **final hidden state** and **cell state** (hidden, cell). These two vectors constitute the **Context Vector**.

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        # Embedding layer: turns word indexes into dense vectors
        self.embedding = nn.Embedding(input_size, hidden_size)

        # LSTM layer
        self.lstm = nn.LSTM(hidden_size, hidden_size)

    def forward(self, input, hidden):
        # input shape: (1, 1) -> (Seq Len, Batch Size)
        embedded = self.embedding(input).view(1, 1, -1)
        output, hidden = self.lstm(embedded, hidden)
        return output, hidden

    def initHidden(self):
        # Initialize hidden and cell states with zeros
        return (torch.zeros(1, 1, self.hidden_size, device=device),
                torch.zeros(1, 1, self.hidden_size, device=device))

## 2.3 The Decoder

The Decoder is also an LSTM.

1.  **Input:** It receives the previous predicted token (starting with SOS) and the previous hidden state.
2.  **Initial State:** Crucially, the **initial hidden state** of the decoder is the **final hidden state** of the Encoder (The Context Vector).
3.  **Linear Layer:** Projects the LSTM output to the size of the vocabulary to predict the next word.

In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        # input is the previous token index
        output = self.embedding(input).view(1, 1, -1)
        output = torch.relu(output)

        # Run LSTM
        output, hidden = self.lstm(output, hidden)

        # Predict next word
        prediction = self.softmax(self.out(output[0]))
        return prediction, hidden

## 2.4 Training the Seq2Seq Model

We train by passing the input sentence through the Encoder, getting the context vector, and passing that to the Decoder.



### 2.4.1 Teacher Forcing
During training, we use a technique called **Teacher Forcing**.
* **Without Teacher Forcing:** The decoder uses its *own predicted word* as input for the next step.
* **With Teacher Forcing:** We feed the decoder the *actual correct word* from the target sentence as input for the next step, regardless of what it predicted. This helps the model learn faster.

In [ ]:
def train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion, max_length=10):
    encoder_hidden = encoder.initHidden()

    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)

    loss = 0

    for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)

    # Decoder's first hidden state is the Encoder's last hidden state (Context Vector)
    decoder_input = torch.tensor([[SOS_token]], device=device)
    decoder_hidden = encoder_hidden

    # Teacher forcing: Feed the target as the next input
    use_teacher_forcing = True if random.random() < 0.5 else False

    if use_teacher_forcing:
        # Teacher forcing: Feed the target as the next input
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]  # Teacher forcing

    else:
        # Without teacher forcing: use its own predictions as the next input
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.detach()  # detach from history as input

            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break

    loss.backward()

    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / target_length

### 2.4.2 The Training Loop
We will run the training for 1000 epochs. Since our dataset is tiny, this will be extremely fast and ensure the model memorizes the patterns.

In [ ]:
hidden_size = 256
encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

learning_rate = 0.01
encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
criterion = nn.NLLLoss()

n_iters = 1000
print_every = 100
plot_losses = []

print("Starting Training...")

for iter in range(1, n_iters + 1):
    training_pair = tensorsFromPair(random.choice(pairs))
    input_tensor = training_pair[0]
    target_tensor = training_pair[1]

    loss = train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)

    if iter % print_every == 0:
        print(f'Iter: {iter} | Loss: {loss:.4f}')

## 2.5 Evaluation (Inference)

To test the model, we perform greedy decoding:
1. Feed the input sentence into the Encoder.
2. Grab the context vector.
3. Feed the SOS token and context vector to the Decoder.
4. At every step, take the word with the highest probability, and feed it into the next step until we hit `<EOS>`.

In [ ]:
def evaluate(encoder, decoder, sentence, max_length=10):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        input_length = input_tensor.size()[0]
        encoder_hidden = encoder.initHidden()

        for ei in range(input_length):
            encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)

        decoder_input = torch.tensor([[SOS_token]], device=device)
        decoder_hidden = encoder_hidden

        decoded_words = []

        for di in range(max_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = decoder_output.data.topk(1)

            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])

            decoder_input = topi.detach()

        return decoded_words

# Test on a few sentences from our training set
def evaluateRandomly(encoder, decoder, n=3):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words = evaluate(encoder, decoder, pair[0])
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

evaluateRandomly(encoder, decoder)

# 3. Conclusion

**Summary of mechanics:**
1.  **Encoder** compressed the English sentence into a vector.
2.  **Decoder** unpacked that vector into French.
3.  **Teacher Forcing** helped stabilize the training.

**Next Steps:**
In future sessions, we will discuss the **Attention Mechanism**, which solves the "bottleneck" problem by allowing the decoder to look back at specific parts of the input sentence, rather than relying solely on the single context vector.